### Text summarization using llama2-chat and llama-cpp-python bindings

In [1]:
import pathlib
import spacy

workpath = pathlib.Path('/mnt/Data/shared/ipp')

# Load models
#nlp = spacy.load("pt_core_news_md")  # Use pt_core_news_lg for a more accurate model

In [2]:
import torch
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/mnt/Data/venvai/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


ValueError: `rope_scaling` must be a dictionary with with two fields, `type` and `factor`, got {'factor': 32.0, 'high_freq_factor': 4.0, 'low_freq_factor': 1.0, 'original_max_position_embeddings': 8192, 'rope_type': 'llama3'}

In [48]:
# Function to tokenize and move tokens to device
def tokenize_and_to_device(text, tokenizer, device):
    tokens = tokenizer(text, return_tensors="pt")
    return {k: v.to(device) for k, v in tokens.items()}

# Function to calculate log-probabilities for the completion
def calculate_logprob(sequence_logits, sequence_tokens, completion_tokens, length_penalty=0.5):
    completion_start_idx = len(sequence_tokens["input_ids"][0]) - len(completion_tokens["input_ids"][0])
    completion_logits = sequence_logits[0, completion_start_idx - 1:-1]
    completion_ids = sequence_tokens["input_ids"][0][completion_start_idx:]

    # Calculate log-probabilities
    log_probs = torch.nn.functional.log_softmax(completion_logits, dim=-1)
    token_log_probs = log_probs[range(len(completion_ids)), completion_ids]
    
    # Normalize by length of completion
    completion_length = len(completion_ids)

    return token_log_probs.sum().item() * (completion_length ** length_penalty)

# Function to evaluate two possible completions
def evaluate_completions(context_fragments, proposed_text, alternative_text, tokenizer, model, device):
    # Get the last 4 fragments as context (or fewer if there are not enough)
    context = " ".join(context_fragments[-4:])

    # Tokenize context and completions
    context_tokens = tokenize_and_to_device(context, tokenizer, device)
    proposed_tokens = tokenize_and_to_device(proposed_text, tokenizer, device)
    alternative_tokens = tokenize_and_to_device(alternative_text, tokenizer, device)

    # Combine context with each completion
    merged_proposed = tokenize_and_to_device(context + proposed_text, tokenizer, device)
    merged_alternative = tokenize_and_to_device(context + alternative_text, tokenizer, device)

    # Compute log-probabilities for each sequence
    with torch.no_grad():
        proposed_outputs = model(**merged_proposed)
        alternative_outputs = model(**merged_alternative)

    # Extract logits
    proposed_logits = proposed_outputs.logits
    alternative_logits = alternative_outputs.logits

    # Calculate log-probabilities for each completion
    proposed_logprob = calculate_logprob(proposed_logits, merged_proposed, proposed_tokens)
    alternative_logprob = calculate_logprob(alternative_logits, merged_alternative, alternative_tokens)

    # Compare likelihoods and return the more likely completion
    if proposed_logprob > alternative_logprob:
        return "The proposed completion is more likely."
    else:
        return "The alternative completion is more likely."

# Example usage:
context_fragments = ["The cat sat", "on", "the mat", "and"]
proposed_merge = "food."
alternative_merge = "."

result = evaluate_completions(context_fragments, proposed_merge, alternative_merge, tokenizer, model, device)
print(result)


The alternative completion is more likely.


In [11]:
proposed_logprob, alternative_logprob

(-12.307526588439941, -6.161680698394775)

In [ ]:
# Example transcription
file = (workpath / 'whisper_fragments'/ '10-10-2021 - Eclesiastes 3.2b-3a.txt')
sentences = file.open('r').readlines()
# doc = nlp(text)
# sentences = [sent.text.strip().replace('\n', ' ') for sent in doc.sents]
sentences

['Música\n',
 'Irmãos, mais uma vez, boa noite.\n',
 'Hoje daremos continuidade com a graça de Deus\n',
 'na exposição do livro de Eclesiastes.\n',
 'Portanto, peço por gentileza que abram suas Bíblias\n',
 'neste maravilhoso livro, no capítulo 3.\n',
 'Hoje, nossa meditação será no verso 2, parte B,\n',
 'e no verso 3, parte A.\n',
 'Eclesiastes 3,\n',
 'verso 2, parte B,\n',
 'verso 3, parte A.\n',
 'Que diz assim,\n',
 'Tempo de plantar\n',
 'e tempo de arrancar o que se plantou.\n',
 'Tempo de matar\n',
 'e tempo de curar.\n',
 'Vamos orar?\n',
 'Senhor Deus e Pai bendito,\n',
 'precioso,\n',
 'sol que está acima do sol,\n',
 'a Ti, ó Deus, toda honra e glória,\n',
 'a Ti, ó Deus, todo poder e força.\n',
 'e mais uma vez suplicamos,\n',
 'precisamos de Ti,\n',
 'mais do que tudo.\n',
 'Estamos diante da Tua Palavra\n',
 'e queremos, ó Deus, conhecer o Senhor cada vez mais,\n',
 'para assim sermos santificados,\n',
 'crescermos,\n',
 'até alcançarmos a estatura do Teu amado Filho.\n